# Actividad 3 - Aprendizaje Supervisado y No Supervisado

**Dataset:** eCommerce behavior data from multi category store (Kaggle)  
**Fuente:** mkechinov/ecommerce-behavior-data-from-multi-category-store  
**Autor:** Antonio Ramón Valerio Tejada - A01797448  
**Herramienta:** PySpark MLlib

## 1. Introducción Teórica

El aprendizaje automático permite identificar patrones en datos para realizar predicciones o descubrir estructuras ocultas. Se divide principalmente en dos enfoques:

### Aprendizaje Supervisado
Los datos incluyen una variable objetivo o etiqueta. El modelo aprende la relación entre variables de entrada y la etiqueta para predecir nuevos casos. Ejemplos: regresión, clasificación.

### Aprendizaje No Supervisado
Los datos no tienen etiqueta definida. El objetivo es descubrir patrones, similitudes o grupos naturales. Ejemplos: clustering, análisis de asociación.

### Preparación de Datos
Antes de entrenar modelos, es necesario:
- Construir muestras representativas
- Limpiar inconsistencias
- Transformar variables
- Dividir datos en entrenamiento y prueba

En Big Data, esto requiere estrategias específicas para volúmenes grandes de información.

## 2. Instalación e Importación de Librerías

In [ ]:
# !pip install pyspark findspark kagglehub pandas matplotlib

import os
import findspark
findspark.init()

import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator

SEED = 42

## 3. Creación de SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Actividad_Supervisado_NoSupervisado_eCommerce")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

## 4. Selección y Carga de Datos

Estrategia para evitar tiempos de procesamiento altos:
1. Descargar dataset de Kaggle
2. Leer archivo mensual (2019-Oct.csv)
3. Tomar fracción pequeña del dataset
4. Generar variables de caracterización
5. Crear particiones por evento y precio
6. Generar muestra equilibrada (M)

In [ ]:
import kagglehub

DATASET_DIR = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")
print("Dataset descargado en:", DATASET_DIR)

csv_path = None
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file == "2019-Oct.csv":
            csv_path = os.path.join(root, file)

if csv_path is None:
    raise FileNotFoundError("No se encontró 2019-Oct.csv")

print("Archivo:", csv_path)

In [ ]:
df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)

print("Columnas:")
print(df_raw.columns)
df_raw.printSchema()
print("\nMuestras iniciales:")
df_raw.show(5, truncate=False)

In [ ]:
BASE_SAMPLE_FRACTION = 0.01
df = df_raw.sample(withReplacement=False, fraction=BASE_SAMPLE_FRACTION, seed=SEED)

print(f"Muestra base creada: {df.count()} registros")
df.show(5, truncate=False)

## 4.1 Limpieza Inicial y Variables de Caracterización

Se generan variables derivadas:
- event_date: fecha del evento
- hour: hora del día
- day_of_week: día de la semana
- category_level_1: categoría principal
- price_segment: segmento de precio (cuantiles)
- is_purchase: variable objetivo binaria

In [ ]:
df_clean = (
    df
    .dropna(subset=["event_time", "event_type", "product_id", "price", "user_id", "user_session"])
    .filter(F.col("price") >= 0)
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.date_format("event_time", "E"))
    .withColumn(
        "category_level_1",
        F.when(F.col("category_code").isNull(), "unknown")
         .otherwise(F.split(F.col("category_code"), "\\.").getItem(0))
    )
    .withColumn("brand", F.when(F.col("brand").isNull(), "unknown").otherwise(F.col("brand")))
)

q25, q50, q75 = df_clean.approxQuantile("price", [0.25, 0.50, 0.75], 0.05)

df_clean = (
    df_clean
    .withColumn(
        "price_segment",
        F.when(F.col("price") <= q25, "bajo")
         .when(F.col("price") <= q50, "medio_bajo")
         .when(F.col("price") <= q75, "medio_alto")
         .otherwise("alto")
    )
    .withColumn("is_purchase", F.when(F.col("event_type") == "purchase", 1.0).otherwise(0.0))
)

print("Datos limpios:")
df_clean.show(5, truncate=False)

In [ ]:
print("\nEstadísticas Generales:")
df_clean.select(
    F.count("*").alias("registros"),
    F.mean("price").alias("precio_promedio"),
    F.min("price").alias("precio_minimo"),
    F.max("price").alias("precio_maximo")
).show()

print("Distribución de event_type:")
df_clean.groupBy("event_type").count().orderBy(F.desc("count")).show()

print("Distribución de price_segment:")
df_clean.groupBy("price_segment").count().orderBy(F.desc("count")).show()

print("Top 10 categorías:")
df_clean.groupBy("category_level_1").count().orderBy(F.desc("count")).show(10)

## 4.2 Generación de Muestra de Trabajo

Se utiliza muestreo estratificado considerando:
- event_type (view, cart, purchase)
- price_segment (bajo, medio_bajo, medio_alto, alto)

Esto asegura representación de todos los grupos incluyendo eventos menos frecuentes.

In [ ]:
df_part = (
    df_clean
    .withColumn(
        "partition_rule",
        F.concat_ws("_", F.col("event_type"), F.col("price_segment"))
    )
)

partition_counts = (
    df_part
    .groupBy("partition_rule")
    .count()
    .orderBy(F.desc("count"))
)

print("Conteos por partición:")
partition_counts.show()

In [ ]:
N_PER_PARTITION = 600

allocation = (
    partition_counts
    .withColumn(
        "fraccion_muestreo",
        F.when(F.col("count") <= N_PER_PARTITION, F.lit(1.0))
         .otherwise(F.lit(N_PER_PARTITION) / F.col("count"))
    )
)

fractions = {
    row["partition_rule"]: min(float(row["fraccion_muestreo"]), 1.0)
    for row in allocation.collect()
}

M = df_part.sampleBy("partition_rule", fractions=fractions, seed=SEED)

print(f"Muestra M creada: {M.count()} registros")
print("\nDistribución por partición:")
M.groupBy("partition_rule").count().orderBy(F.desc("count")).show()

print("\nDistribución de variable objetivo:")
M.groupBy("is_purchase").count().show()

## 5. Preparación para Modelado

Se realiza:
- Eliminación de registros incompletos
- Tratamiento de valores faltantes en categóricas
- Codificación de variables categóricas
- Creación de vector de características
- Definición de variable objetivo

In [ ]:
model_cols = [
    "price",
    "hour",
    "event_type",
    "category_level_1",
    "brand",
    "price_segment",
    "day_of_week",
    "is_purchase",
    "partition_rule"
]

M_model = (
    M
    .select(*model_cols)
    .dropna(subset=["price", "hour", "is_purchase"])
)

top_brands = [
    row["brand"]
    for row in M_model.groupBy("brand").count().orderBy(F.desc("count")).limit(25).collect()
]

M_model = M_model.withColumn(
    "brand_limited",
    F.when(F.col("brand").isin(top_brands), F.col("brand")).otherwise("other")
)

print("Muestra lista para modelado:")
M_model.select(
    "price", "hour", "event_type", "category_level_1",
    "brand_limited", "price_segment", "day_of_week", "is_purchase"
).show(5)

In [ ]:
target_distribution = (
    M_model
    .groupBy("is_purchase")
    .count()
    .withColumn("porcentaje", F.round(F.col("count") / F.sum("count").over(Window.partitionBy()) * 100, 2))
)

print("Distribución de variable objetivo:")
target_distribution.show()

## 6. División Entrenamiento y Prueba

Proporción: 80/20 con estratificación por is_purchase

Se calculan pesos de clase para compensar desbalance.

In [ ]:
M_indexed = M_model.withColumn("row_id", F.monotonically_increasing_id())

train_fraction_by_class = {0.0: 0.80, 1.0: 0.80}

train = M_indexed.sampleBy(
    "is_purchase",
    fractions=train_fraction_by_class,
    seed=SEED
)

train_ids = train.select("row_id").withColumn("in_train", F.lit(1))

test = (
    M_indexed
    .join(train_ids, on="row_id", how="left")
    .filter(F.col("in_train").isNull())
    .drop("in_train")
)

print(f"Train: {train.count()} registros")
print(f"Test: {test.count()} registros")

print("\nDistribución en entrenamiento:")
train.groupBy("is_purchase").count().show()

print("Distribución en prueba:")
test.groupBy("is_purchase").count().show()

In [ ]:
class_counts = {
    row["is_purchase"]: row["count"]
    for row in train.groupBy("is_purchase").count().collect()
}

n_train = sum(class_counts.values())

weight_0 = n_train / (2 * class_counts.get(0.0, 1))
weight_1 = n_train / (2 * class_counts.get(1.0, 1))

train = train.withColumn(
    "class_weight",
    F.when(F.col("is_purchase") == 1.0, F.lit(weight_1)).otherwise(F.lit(weight_0))
)

test = test.withColumn(
    "class_weight",
    F.when(F.col("is_purchase") == 1.0, F.lit(weight_1)).otherwise(F.lit(weight_0))
)

print(f"Peso clase 0 (no compra): {weight_0:.4f}")
print(f"Peso clase 1 (compra): {weight_1:.4f}")

## 7. Modelo de Clasificación - Random Forest

Predice si una interacción termina en compra (is_purchase = 1) o no.

Características:
- 80 árboles
- Profundidad máxima: 7
- Pesos de clase para balanceo
- Excluye event_type (generada de la variable objetivo)

In [ ]:
categorical_cols = [
    "category_level_1",
    "brand_limited",
    "price_segment",
    "day_of_week"
]

numeric_cols = ["price", "hour"]

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{col}_idx" for col in categorical_cols],
    outputCols=[f"{col}_ohe" for col in categorical_cols],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{col}_ohe" for col in categorical_cols],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    labelCol="is_purchase",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=80,
    maxDepth=7,
    minInstancesPerNode=5,
    seed=SEED
)

rf_pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])

print("Entrenando modelo Random Forest...")
rf_model = rf_pipeline.fit(train)

print("Realizando predicciones...")
predictions = rf_model.transform(test)

print("\nPredicciones:")
predictions.select(
    "is_purchase", "prediction", "probability",
    "price", "hour", "category_level_1", "brand_limited", "price_segment"
).show(10)

## 8. Evaluación del Modelo

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("Evaluación del modelo Random Forest:")
print("="*50)

binary_evaluator = BinaryClassificationEvaluator(
    labelCol="is_purchase",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = binary_evaluator.evaluate(predictions)
print(f"AUC-ROC: {auc:.4f}")

multiclass_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_purchase",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = multiclass_evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_purchase",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = f1_evaluator.evaluate(predictions)
print(f"Precision: {precision:.4f}")

## Conclusiones

Este notebook implementa un pipeline completo de aprendizaje automático usando PySpark MLlib:

1. **Carga y exploración** de datos de comportamiento de eCommerce
2. **Limpieza y transformación** de variables
3. **Ingeniería de características** incluyendo segmentación por precio
4. **Muestreo estratificado** para garantizar representación balanceada
5. **Entrenamiento** de un modelo Random Forest para predicción de compras
6. **Evaluación** del desempeño del modelo

El modelo puede utilizarse para identificar interacciones de usuarios con alta probabilidad de conversión a compra.